# 00 Workflow Overview

This notebook is a workflow guide and artifact status board.
It does **not** execute notebooks `01-06` automatically.

## What happens here
- checks which pipeline artifacts already exist
- helps decide which notebook should be executed next

## Pipeline map
| Step | Notebook | Input | Main output |
| --- | --- | --- | --- |
| 1 | `01_normalize.ipynb` | ingested logs | `normalized/flows.jsonl` |
| 2 | `02_inventory.ipynb` | `normalized/flows.jsonl` | `inventory/hosts.jsonl` |
| 3 | `03_enrich.ipynb` | `normalized/flows.jsonl` | `enriched/enriched_hosts.jsonl` |
| 4 | `04_analyze_graph.ipynb` | normalized + inventory/enriched | `graph/graph.json` |
| 5 | `05_criticality_export.ipynb` | `graph/graph.json` | `criticality/*.jsonl`, `report/*` |
| 6 | `06_report_dashboard.ipynb` | existing artifacts | fast report regeneration |

## How to use
1. Open the next notebook in order.
2. Run the step using the control panel buttons.
3. Return here and refresh artifact status.


### Environment initialization
This cell sets the project root and shared `RUN_DIR` used by all notebooks.


In [ ]:
from pathlib import Path
import os
import sys


def find_project_root(start: Path) -> Path:
    candidates = [start, *start.parents]

    env_root = os.environ.get("PMMAP_ROOT")
    if env_root:
        candidates.append(Path(env_root))

    candidates.append(Path.home() / "Bakalarka" / "passive-network-mapping-platform")

    seen = set()
    for path in candidates:
        try:
            path = path.resolve()
        except Exception:
            pass
        key = str(path)
        if key in seen:
            continue
        seen.add(key)
        if (path / "pmmap").is_dir():
            return path

    raise RuntimeError(
        "Could not locate project root (directory containing 'pmmap'). "
        "Open notebook from repository workspace or set PMMAP_ROOT."
    )


ROOT = find_project_root(Path().resolve())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pmmap.notebook_common import init_notebook_paths

ROOT, DATA_DIR, RUN_DIR = init_notebook_paths(start=ROOT)


### Artifact status panel
Use this panel to verify which outputs are already ready and which step is still missing.


In [ ]:
from pmmap.notebook_interactive import create_artifact_overview_controls

create_artifact_overview_controls(RUN_DIR)
